# 010 · 토크나이저 — 텍스트가 모델에 들어가는 형태

**1일차 3교시** · 슬라이드 12–17 · CPU로 실행 가능

## 이 노트북에서 하는 일
1. 형태소 분석기(KoNLPy)와 subword 토크나이저를 같은 문장에 적용해 비교한다
2. BPE · WordPiece · Unigram이 같은 단어를 어떻게 다르게 쪼개는지 관찰한다
3. **특수 토큰**을 확인한다 — 2일차 chat template의 전제다
4. 어휘 사전에 없는 말(신조어·오탈자)이 어떻게 처리되는지 본다

## 개편 변경사항
- `tf.keras.preprocessing.text.Tokenizer`는 **삭제**했다. TensorFlow 2.16+에서 deprecated 경로로 밀렸고,
  이 과정의 스택은 PyTorch + HuggingFace다.
- HuggingFace `AutoTokenizer`를 중심으로 재구성했다.

In [1]:
# --- 패키지 설치 (Colab) ---
# 설치 후 "런타임 다시 시작" 안내가 나오면 재시작하고, 다음 셀부터 이어서 실행한다.
!pip install -q transformers==4.57.1 datasets==4.0.0 accelerate==1.10.1 \
    peft==0.17.1 trl==0.23.0 bitsandbytes==0.47.0 tiktoken==0.11.0
!apt-get install -y fonts-nanum -qq > /dev/null   # matplotlib 한글 폰트

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 59.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 374.9/374.9 kB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 504.9/504.9 kB 35.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 564.7/564.7 kB 25.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 MB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 52.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 24.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.20.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [2]:
# --- 저장소 클론 및 루트 이동 (Colab) ---
from pathlib import Path

if not Path("common/config.py").exists():
    !git clone -q https://github.com/ironmanciti/LLM_FineTuning_Lecture.git
    %cd LLM_FineTuning_Lecture
print("저장소 루트:", Path.cwd())

from common import config as C, env, artifacts as art
env.set_seed(C.SEED)

/content/LLM_FineTuning_Lecture
저장소 루트: /content/LLM_FineTuning_Lecture


42

---
## 1. 같은 문장, 세 가지 관점

한국어는 **교착어**다. 어간에 조사·어미가 붙어 한 어절이 만들어진다.
`먹었습니다` 하나에 어간 `먹` + 과거 `었` + 종결 `습니다`가 들어 있다.

| 관점 | 도구 | 단위 |
|---|---|---|
| 형태소 | KoNLPy (Okt, Mecab) | 언어학적 최소 의미 단위 |
| subword | BPE / WordPiece / Unigram | **데이터에서 통계적으로 학습된** 조각 |
| 문자 | `list(text)` | 음절 |

LLM은 전부 subword를 쓴다. 형태소 분석기를 배우는 이유는 두 가지다 —
전통 NLP 파이프라인의 이해, 그리고 **키워드 검색(BM25)의 한국어 토큰화**에 쓰이기 때문이다.

In [3]:
SENT = "딥러닝을 활용한 자연어처리 과정에서 LoRA로 파인튜닝을 해봤습니다."

print("원문     :", SENT)
print("길이     :", len(SENT), "자")
print("공백분리 :", SENT.split())
print("음절     :", list(SENT)[:20], "...")

원문     : 딥러닝을 활용한 자연어처리 과정에서 LoRA로 파인튜닝을 해봤습니다.
길이     : 38 자
공백분리 : ['딥러닝을', '활용한', '자연어처리', '과정에서', 'LoRA로', '파인튜닝을', '해봤습니다.']
음절     : ['딥', '러', '닝', '을', ' ', '활', '용', '한', ' ', '자', '연', '어', '처', '리', ' ', '과', '정', '에', '서', ' '] ...


In [4]:
# KoNLPy Okt 형태소 분석 — Colab 기본 JDK 위에 konlpy만 설치하면 된다
!pip install -q konlpy

from konlpy.tag import Okt
okt = Okt()
print("형태소   :", okt.morphs(SENT))
print("품사     :", okt.pos(SENT)[:12], "...")
print("명사     :", okt.nouns(SENT))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.4/19.4 MB 92.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 438.5/438.5 kB 39.9 MB/s eta 0:00:00
형태소   : ['딥', '러닝', '을', '활용', '한', '자연어', '처리', '과정', '에서', 'LoRA', '로', '파인튜닝', '을', '해봤습니다', '.']
품사     : [('딥', 'Noun'), ('러닝', 'Noun'), ('을', 'Josa'), ('활용', 'Noun'), ('한', 'Josa'), ('자연어', 'Noun'), ('처리', 'Noun'), ('과정', 'Noun'), ('에서', 'Josa'), ('LoRA', 'Alpha'), ('로', 'Noun'), ('파인튜닝', 'Noun')] ...
명사     : ['딥', '러닝', '활용', '자연어', '처리', '과정', '로', '파인튜닝']


---
## 2. subword 토크나이저 3종 비교

같은 문장을 세 계열로 잘라 본다.

| 알고리즘 | 대표 모델 계열 | 학습 방식 |
|---|---|---|
| **BPE** | GPT, Qwen, Llama | 자주 함께 나오는 쌍을 반복 병합 |
| **WordPiece** | BERT, KLUE-RoBERTa | 우도(likelihood)를 가장 높이는 쌍을 병합. 이어지는 조각에 `##` |
| **Unigram** (SentencePiece) | Gemma, T5 | 큰 어휘에서 시작해 손실이 가장 적은 조각을 **제거** |

읽는 법: `Ġ`(BPE)와 `▁`(SentencePiece)는 **앞에 공백이 있었다**는 표시다.
`##`(WordPiece)는 **앞 조각에 이어 붙는다**는 표시다.

In [5]:
from transformers import AutoTokenizer

# config.TOKENIZER_ZOO에서 가져온다. 접근이 막힌 모델은 자동으로 건너뛴다.
loaded = {}
for label, ref in C.TOKENIZER_ZOO.items():
    if label.startswith("tiktoken:"):
        continue
    try:
        loaded[label] = AutoTokenizer.from_pretrained(ref, trust_remote_code=False)
        print(f"  [OK]   {label:<14} {ref}")
    except Exception as e:
        print(f"  [건너뜀] {label:<14} {type(e).__name__} — gated 모델이면 HF 토큰이 필요합니다")

print(f"\n{len(loaded)}종 로드 완료")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/375 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/173 [00:00<?, ?B/s]

  [OK]   KLUE-RoBERTa   klue/roberta-base

2종 로드 완료


In [6]:
def show(tok, label, text):
    ids = tok(text, add_special_tokens=False)["input_ids"]
    pieces = tok.convert_ids_to_tokens(ids)
    print(f"\n── {label}  (vocab {tok.vocab_size:,} / {len(ids)}토큰)")
    print("   ", " │ ".join(pieces))

for label, tok in loaded.items():
    show(tok, label, SENT)


── Qwen2.5  (vocab 151,643 / 25토큰)
    ëĶ¥ │ ëŁ¬ │ ëĭĿ │ ìĿĦ │ ĠíĻľìļ© │ íķľ │ ĠìŀĲìĹ° │ ìĸ´ │ ì²ĺë¦¬ │ Ġê³ │ ¼ │ ìłķ │ ìĹĲìĦľ │ ĠLo │ RA │ ë¡ľ │ ĠíĮĮ │ ìĿ¸ │ íĬľ │ ëĭĿ │ ìĿĦ │ Ġíķ´ │ ë´¤ │ ìĬµëĭĪëĭ¤ │ .

── KLUE-RoBERTa  (vocab 32,000 / 24토큰)
    딥 │ ##러 │ ##닝 │ ##을 │ 활용 │ ##한 │ 자연 │ ##어 │ ##처리 │ 과정 │ ##에서 │ L │ ##o │ ##RA │ ##로 │ 파인 │ ##튜 │ ##닝 │ ##을 │ 해 │ ##봤 │ ##습 │ ##니다 │ .


### 확인할 것

- **`파인튜닝`, `LoRA` 같은 말이 몇 조각으로 쪼개지는가.** 영어권 데이터로 학습된
  토크나이저는 한국어 신조어를 잘게 부순다. 이것이 다음 노트북(`020`)의 주제다.
- **어휘 크기가 클수록 조각이 적다.** 다만 어휘가 커지면 임베딩 행렬과 LM head가
  커진다(vocab × hidden). 0.5B 모델에서 어휘가 15만이면 임베딩만 전체의 상당 부분을 차지한다.

In [7]:
# 어휘 크기와 임베딩 파라미터 비중 — 슬라이드 57(LM head)의 복선
HIDDEN = 896   # Qwen2.5-0.5B의 hidden_size
print(f"{'토크나이저':<16}{'vocab':>10}{'임베딩 파라미터':>18}")
for label, tok in loaded.items():
    n = tok.vocab_size * HIDDEN
    print(f"{label:<16}{tok.vocab_size:>10,}{n/1e6:>15.1f}M")
print("\n※ weight tying(입력 임베딩과 LM head 공유)을 쓰면 이 값이 두 번 들지 않는다.")

토크나이저                vocab          임베딩 파라미터
Qwen2.5            151,643          135.9M
KLUE-RoBERTa        32,000           28.7M

※ weight tying(입력 임베딩과 LM head 공유)을 쓰면 이 값이 두 번 들지 않는다.


---
## 3. BPE 병합 과정 직접 보기

BPE는 원래 **데이터 압축** 알고리즘이다. 가장 자주 붙어 나오는 쌍을 하나로 합치고,
그 과정을 정해진 횟수만큼 반복한다. 학습된 병합 규칙의 순서가 곧 어휘 사전이다.

In [8]:
from collections import Counter

def bpe_demo(words, n_merges=8):
    """작은 코퍼스로 BPE 병합을 직접 돌려 본다."""
    vocab = {" ".join(list(w)) + " </w>": c for w, c in Counter(words).items()}
    for step in range(1, n_merges + 1):
        pairs = Counter()
        for word, freq in vocab.items():
            syms = word.split()
            for i in range(len(syms) - 1):
                pairs[(syms[i], syms[i + 1])] += freq
        if not pairs:
            break
        best, cnt = pairs.most_common(1)[0]
        vocab = {w.replace(" ".join(best), "".join(best)): c for w, c in vocab.items()}
        print(f"  {step:>2}. {best[0]!r} + {best[1]!r} -> {''.join(best)!r}   (빈도 {cnt})")
    return vocab

corpus = ["파인튜닝", "파인튜닝", "파인튜닝", "튜닝", "튜닝", "미세튜닝", "파인", "파인"]
print("코퍼스:", corpus)
print("\n병합 순서:")
final = bpe_demo(corpus, n_merges=8)
print("\n최종 분절:")
for w, c in final.items():
    print("  ", w)

코퍼스: ['파인튜닝', '파인튜닝', '파인튜닝', '튜닝', '튜닝', '미세튜닝', '파인', '파인']

병합 순서:
   1. '튜' + '닝' -> '튜닝'   (빈도 6)
   2. '튜닝' + '</w>' -> '튜닝</w>'   (빈도 6)
   3. '파' + '인' -> '파인'   (빈도 5)
   4. '파인' + '튜닝</w>' -> '파인튜닝</w>'   (빈도 3)
   5. '파인' + '</w>' -> '파인</w>'   (빈도 2)
   6. '미' + '세' -> '미세'   (빈도 1)
   7. '미세' + '튜닝</w>' -> '미세튜닝</w>'   (빈도 1)

최종 분절:
   파인튜닝</w>
   튜닝</w>
   미세튜닝</w>
   파인</w>


`튜닝`이 자주 함께 나오므로 하나의 조각으로 승격된다. **자주 쓰이는 말은 통째로,
드문 말은 잘게** — 이것이 subword의 핵심이고, 다음 노트북에서 이 성질이 한국어에
어떻게 불리하게 작동하는지 본다.

---
## 4. 특수 토큰 — 2일차의 전제

파인튜닝에서 가장 자주 발생하는 실패가 **특수 토큰을 직접 문자열로 조립하는 것**이다.
모델마다 다르므로 반드시 토크나이저에게 물어야 한다.

In [9]:
for label, tok in loaded.items():
    print(f"\n── {label}")
    print(f"   bos={tok.bos_token!r}  eos={tok.eos_token!r}  pad={tok.pad_token!r}  unk={tok.unk_token!r}")
    extra = [t for t in (tok.additional_special_tokens or [])][:6]
    if extra:
        print(f"   추가 특수 토큰: {extra}")
    has_tpl = getattr(tok, "chat_template", None) is not None
    print(f"   chat_template 보유: {has_tpl}")


── Qwen2.5
   bos=None  eos='<|im_end|>'  pad='<|endoftext|>'  unk=None
   추가 특수 토큰: ['<|im_start|>', '<|im_end|>', '<|object_ref_start|>', '<|object_ref_end|>', '<|box_start|>', '<|box_end|>']
   chat_template 보유: True

── KLUE-RoBERTa
   bos='[CLS]'  eos='[SEP]'  pad='[PAD]'  unk='[UNK]'
   chat_template 보유: False


In [10]:
# chat_template이 실제로 무엇을 만드는지 미리 한 번 본다 (2일차 §3.9-3의 예고)
from common import chat

tok = loaded.get("Qwen2.5") or next(iter(loaded.values()))
msgs = chat.build_messages("대한민국의 수도는 어디입니까?", system="간결하게 답하십시오.")
rendered = chat.render_prompt(tok, msgs)
print(repr(rendered))
print("\n" + "=" * 62)
print(rendered)

'<|im_start|>system\n간결하게 답하십시오.<|im_end|>\n<|im_start|>user\n대한민국의 수도는 어디입니까?<|im_end|>\n<|im_start|>assistant\n'

<|im_start|>system
간결하게 답하십시오.<|im_end|>
<|im_start|>user
대한민국의 수도는 어디입니까?<|im_end|>
<|im_start|>assistant



**`<|im_start|>`, `<|im_end|>` 같은 토큰이 자동으로 들어간 것을 확인하십시오.**
이 문자열을 손으로 쓰면 모델·버전이 바뀔 때마다 조용히 깨진다.

---
## 5. 어휘에 없는 말

`UNK`로 떨어지는가, 조각으로 쪼개지는가? subword의 이점이 여기서 드러난다.

In [11]:
UNSEEN = ["QLoRA", "쿼로라", "머신러닝했슴다", "ㅋㅋㅋ루삥뽕", "2026년도예산안", "🤗"]

for label, tok in loaded.items():
    print(f"\n── {label}")
    for w in UNSEEN:
        ids = tok(w, add_special_tokens=False)["input_ids"]
        pieces = tok.convert_ids_to_tokens(ids)
        unk = " ⚠UNK포함" if tok.unk_token in pieces else ""
        print(f"   {w:<14} {len(ids):>2}조각  {pieces}{unk}")


── Qwen2.5
   QLoRA           3조각  ['QL', 'o', 'RA']
   쿼로라             3조각  ['ì¿¼', 'ë¡ľ', 'ëĿ¼']
   머신러닝했슴다         7조각  ['ë¨¸', 'ìĭł', 'ëŁ¬', 'ëĭĿ', 'íĸĪ', 'ìĬ´', 'ëĭ¤']
   ㅋㅋㅋ루삥뽕          6조각  ['ãħĭ', 'ãħĭ', 'ãħĭ', 'ë£¨', 'ìĤ¥', 'ë½ķ']
   2026년도예산안       9조각  ['2', '0', '2', '6', 'ëħĦ', 'ëıĦ', 'ìĺĪ', 'ìĤ°', 'ìķĪ']
   🤗               1조각  ['ðŁ¤Ĺ']

── KLUE-RoBERTa
   QLoRA           4조각  ['Q', '##L', '##o', '##RA']
   쿼로라             2조각  ['쿼', '##로라']
   머신러닝했슴다         6조각  ['머신', '##러', '##닝', '##했', '##슴', '##다']
   ㅋㅋㅋ루삥뽕          1조각  ['[UNK]'] ⚠UNK포함
   2026년도예산안       6조각  ['202', '##6', '##년', '##도', '##예산', '##안']
   🤗               1조각  ['[UNK]'] ⚠UNK포함


---
## 6. tiktoken — API 모델의 토크나이저

OpenAI 계열 모델의 토큰 수를 세려면 `tiktoken`을 쓴다. `020`에서 비용 비교에 사용한다.

In [12]:
try:
    import tiktoken
    enc = tiktoken.get_encoding("cl100k_base")
    ids = enc.encode(SENT)
    print(f"cl100k_base: {len(ids)}토큰")
    print("조각:", [enc.decode([i]) for i in ids])
except Exception as e:
    print(f"[건너뜀] tiktoken: {e}")

cl100k_base: 37토큰
조각: ['�', '�', '러', '�', '�', '을', ' �', '�', '용', '한', ' 자', '�', '�', '어', '�', '�', '리', ' �', '�', '정', '에서', ' Lo', 'RA', '로', ' 파', '인', '�', '�', '�', '�', '을', ' 해', '�', '�', '�', '습니다', '.']


---
## 정리 — 다음으로

| 관찰한 것 | 다음 노트북에서 |
|---|---|
| 한국어가 여러 조각으로 쪼개진다 | `020` 토큰 수를 **측정**하고 학습 설정에 반영한다 |
| 모델마다 특수 토큰이 다르다 | `410` `apply_chat_template`으로 데이터를 만든다 |
| 어휘 크기가 임베딩 크기를 정한다 | `300` 모델을 열어 실제 파라미터 구성을 확인한다 |

### 제출
이 노트북에는 제출물이 없다. 다만 **자기 업무 문장 3개를 `SENT`에 넣어 실행**해 보고,
어느 토크나이저가 자기 도메인 용어를 가장 적게 쪼개는지 기록해 두십시오.
`020`의 결론과 대조합니다.